# Notebook 8: Final Report Generation

## Purpose
Assemble final outputs into publishable format.

## Includes
- Summary statistics
- Key figures and statistical tables
- Interpretation of hypotheses
- Save as `.docx` and `.pdf`

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# For report generation
try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    HAS_DOCX = True
except ImportError:
    print("python-docx not available. Install with: pip install python-docx")
    HAS_DOCX = False

np.random.seed(42)

## 1. Load All Results

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
RESULTS_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"
OUTPUT_DIR = RESULTS_DIR

# Load all result files
results = {}

# Kruskal-Wallis results
kw_file = RESULTS_DIR / "kruskal_wallis_results.csv"
if kw_file.exists():
    results['kruskal_wallis'] = pd.read_csv(kw_file)
    print(f"✓ Loaded Kruskal-Wallis results")

# Post-hoc results
posthoc_file = RESULTS_DIR / "posthoc_pairwise_results.csv"
if posthoc_file.exists():
    results['posthoc'] = pd.read_csv(posthoc_file)
    print(f"✓ Loaded post-hoc results")

# Model coefficients
coef_file = RESULTS_DIR / "model_coefficients.csv"
if coef_file.exists():
    results['coefficients'] = pd.read_csv(coef_file)
    print(f"✓ Loaded model coefficients")

# Timecourse ANOVA
timecourse_file = RESULTS_DIR / "timecourse_anova_results.csv"
if timecourse_file.exists():
    results['timecourse'] = pd.read_csv(timecourse_file)
    print(f"✓ Loaded timecourse results")

# Bootstrap CIs
bootstrap_file = RESULTS_DIR / "bootstrap_confidence_intervals.csv"
if bootstrap_file.exists():
    results['bootstrap'] = pd.read_csv(bootstrap_file)
    print(f"✓ Loaded bootstrap results")

print(f"\nLoaded {len(results)} result files")

## 2. Generate Summary Statistics

In [ ]:
# Create summary of hypothesis tests
summary = []

if 'kruskal_wallis' in results:
    kw = results['kruskal_wallis']
    for _, row in kw.iterrows():
        summary.append({
            'Hypothesis': row.get('hypothesis', 'N/A'),
            'Index': row.get('variable', 'N/A'),
            'Test': 'Kruskal-Wallis',
            'Statistic': f"H={row['h_statistic']:.3f}",
            'P-value': f"{row['p_value']:.4f}",
            'Significant': 'Yes' if row['p_value'] < 0.05 else 'No',
            'Effect Size (η²)': f"{row['eta_squared']:.3f}",
            'Expected Direction': row.get('expected_direction', 'N/A')
        })

summary_df = pd.DataFrame(summary)
print("Summary of Hypothesis Tests:")
print(summary_df.to_string(index=False))

# Save summary
summary_file = OUTPUT_DIR / "hypothesis_test_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"\n✓ Saved: {summary_file}")

## 3. Generate Word Document Report

In [ ]:
if HAS_DOCX:
    doc = Document()
    
    # Title
    title = doc.add_heading('Statistical Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Introduction
    doc.add_heading('1. Executive Summary', 1)
    doc.add_paragraph(
        'This report presents the results of statistical analyses testing hypotheses H1-H6 '
        'regarding differences in thematic content between high-rated (Top) and low-rated (Trash) '
        'billionaire romance novels.'
    )
    
    # Hypothesis Results
    doc.add_heading('2. Hypothesis Test Results', 1)
    
    if len(summary_df) > 0:
        # Create table
        table = doc.add_table(rows=1, cols=len(summary_df.columns))
        table.style = 'Light Grid Accent 1'
        
        # Header row
        header_cells = table.rows[0].cells
        for i, col in enumerate(summary_df.columns):
            header_cells[i].text = col
        
        # Data rows
        for _, row in summary_df.iterrows():
            row_cells = table.add_row().cells
            for i, col in enumerate(summary_df.columns):
                row_cells[i].text = str(row[col])
    
    # Key Findings
    doc.add_heading('3. Key Findings', 1)
    
    if 'kruskal_wallis' in results:
        kw = results['kruskal_wallis']
        significant = kw[kw['p_value'] < 0.05]
        
        doc.add_paragraph(f"\nTotal hypotheses tested: {len(kw)}")
        doc.add_paragraph(f"Significant results (p < 0.05): {len(significant)}")
        
        if len(significant) > 0:
            doc.add_paragraph("\nSignificant findings:")
            for _, row in significant.iterrows():
                doc.add_paragraph(
                    f"  • {row.get('hypothesis', 'N/A')}: {row.get('variable', 'N/A')} "
                    f"(p = {row['p_value']:.4f}, η² = {row['eta_squared']:.3f})",
                    style='List Bullet'
                )
    
    # Save document
    doc_file = OUTPUT_DIR / "statistical_analysis_report.docx"
    doc.save(str(doc_file))
    print(f"\n✓ Saved Word document: {doc_file}")
else:
    print("\n⚠ Word document generation skipped (python-docx not available)")

## 4. Generate Markdown Summary

In [ ]:
# Create markdown summary
md_content = """# Statistical Analysis Report

## Executive Summary

This report presents the results of statistical analyses testing hypotheses H1-H6 regarding differences in thematic content between high-rated (Top) and low-rated (Trash) billionaire romance novels.

## Hypothesis Test Results

"""

if len(summary_df) > 0:
    md_content += summary_df.to_markdown(index=False)
    md_content += "\n\n"

md_content += "## Key Findings\n\n"

if 'kruskal_wallis' in results:
    kw = results['kruskal_wallis']
    significant = kw[kw['p_value'] < 0.05]
    
    md_content += f"- Total hypotheses tested: {len(kw)}\n"
    md_content += f"- Significant results (p < 0.05): {len(significant)}\n\n"
    
    if len(significant) > 0:
        md_content += "### Significant Findings:\n\n"
        for _, row in significant.iterrows():
            md_content += (
                f"- **{row.get('hypothesis', 'N/A')}**: {row.get('variable', 'N/A')} "
                f"(p = {row['p_value']:.4f}, η² = {row['eta_squared']:.3f})\n"
            )

# Save markdown
md_file = OUTPUT_DIR / "statistical_analysis_report.md"
with open(md_file, 'w') as f:
    f.write(md_content)
print(f"\n✓ Saved Markdown report: {md_file}")

## 5. Compile All Figures

In [ ]:
# List all generated figures
eda_dir = RESULTS_DIR / "eda"
figures = []

if eda_dir.exists():
    figures.extend(list(eda_dir.glob("*.png")))

figures.extend(list(RESULTS_DIR.glob("*.png")))

print(f"\nFound {len(figures)} figure files:")
for fig in sorted(figures):
    print(f"  - {fig.name}")

# Create figure index
figure_index = pd.DataFrame({
    'filename': [f.name for f in figures],
    'path': [str(f.relative_to(RESULTS_DIR)) for f in figures]
})

index_file = OUTPUT_DIR / "figure_index.csv"
figure_index.to_csv(index_file, index=False)
print(f"\n✓ Saved figure index: {index_file}")

## Summary

Final report generation complete. All results compiled and saved.